In [26]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

current_year = 2026
starting_year = 2012  # Sempre pega os últimos 10 anos

data = []  # Lista para armazenar todas as informações

# Loop para pegar os últimos 10 anos até os próximos 21 anos
for year in range(starting_year, current_year+1):

    formatted_year = f"{year}"  # Define o formato correto do ano no URL

    if formatted_year == (str(current_year) + 'mid'):
        continue
        
    url = f"https://www.numbeo.com/quality-of-life/rankings_by_country.jsp?title={formatted_year}"
    
    response = requests.get(url)

    response.raise_for_status()
    time.sleep(1)

    soup = BeautifulSoup(response.text, 'html.parser')
    
    table = soup.find('tbody')
    if not table:  # Caso a página não tenha dados, pula para o próximo ano
        continue
    
    headers = ["Rank", "Country", "Quality of Life Index", "Purchasing Power Index", 
                "Safety Index", "Health Care Index", "Cost of Living Index", 
                "Property Price to Income Ratio", "Traffic Commute Time Index", 
                "Pollution Index", "Climate Index", "Year"]

    rank = 1  # Inicializa o ranking para cada ano
    
    for row in table.find_all('tr'):
        columns = row.find_all('td')
    
        # Se houver pelo menos uma coluna e a primeira estiver vazia, usá-la como Rank
        if columns and columns[0].text.strip() == "":
            rank_value = rank  # Define o Rank manualmente
            columns = columns[1:]  # Remove a primeira coluna vazia
        else:
            rank_value = rank  # Continua o rank normal
        
        # Ajustar número de colunas para evitar erro de dimensão
        if len(columns) == len(headers) - 2:  # -2 porque adicionamos Rank e Year manualmente
            row_data = [rank_value] + [col.text.strip() for col in columns]  # Adiciona Rank
            
            # Ajusta o ano para exibição correta
            row_data.append(year)  # Mantém o ano normal
            
            data.append(row_data)
            rank += 1  # Incrementa o ranking

df = pd.DataFrame(data, columns=headers)




In [27]:
df.head()

,Rank,Country,Quality of Life Index,Purchasing Power Index,Safety Index,Health Care Index,Cost of Living Index,Property Price to Income Ratio,Traffic Commute Time Index,Pollution Index,Climate Index,Year
0,1,Switzerland,206.2,138.1,68.2,68.0,143.9,7.1,24.6,26.8,-,2014
1,2,United States,195.5,132.9,49.9,68.6,77.4,2.4,37.1,35.1,-,2014
2,3,Germany,192.7,112.3,72.9,75.3,87.1,5.6,36.1,30.2,-,2014
3,4,Sweden,180.9,106.1,61.7,75.2,103.7,9.4,34.1,17.5,-,2014
4,5,Finland,178.9,97.9,70.8,68.7,103.3,7.8,37.0,16.7,-,2014


In [28]:
df_qol = df.copy()

In [29]:
df_qol.replace({'-': pd.NA}, inplace=True)

In [30]:
df_qol.head()

,Rank,Country,Quality of Life Index,Purchasing Power Index,Safety Index,Health Care Index,Cost of Living Index,Property Price to Income Ratio,Traffic Commute Time Index,Pollution Index,Climate Index,Year
0,1,Switzerland,206.2,138.1,68.2,68.0,143.9,7.1,24.6,26.8,<NA>,2014
1,2,United States,195.5,132.9,49.9,68.6,77.4,2.4,37.1,35.1,<NA>,2014
2,3,Germany,192.7,112.3,72.9,75.3,87.1,5.6,36.1,30.2,<NA>,2014
3,4,Sweden,180.9,106.1,61.7,75.2,103.7,9.4,34.1,17.5,<NA>,2014
4,5,Finland,178.9,97.9,70.8,68.7,103.3,7.8,37.0,16.7,<NA>,2014


In [31]:
df_qol["Quality of Life Index"] = pd.to_numeric(df_qol["Quality of Life Index"], errors='coerce')
df_qol["Purchasing Power Index"] = pd.to_numeric(df_qol["Purchasing Power Index"], errors='coerce')
df_qol["Safety Index"] = pd.to_numeric(df_qol["Safety Index"], errors='coerce')
df_qol["Health Care Index"] = pd.to_numeric(df_qol["Health Care Index"], errors='coerce')
df_qol["Cost of Living Index"] = pd.to_numeric(df_qol["Cost of Living Index"], errors='coerce')
df_qol["Property Price to Income Ratio"] = pd.to_numeric(df_qol["Property Price to Income Ratio"], errors='coerce')
df_qol["Traffic Commute Time Index"] = pd.to_numeric(df_qol["Traffic Commute Time Index"], errors='coerce')
df_qol["Pollution Index"] = pd.to_numeric(df_qol["Pollution Index"], errors='coerce')
df_qol["Climate Index"] = pd.to_numeric(df_qol["Climate Index"], errors='coerce') 


In [32]:
df_qol.head()

,Rank,Country,Quality of Life Index,Purchasing Power Index,Safety Index,Health Care Index,Cost of Living Index,Property Price to Income Ratio,Traffic Commute Time Index,Pollution Index,Climate Index,Year
0,1,Switzerland,206.2,138.1,68.2,68.0,143.9,7.1,24.6,26.8,NaN,2014
1,2,United States,195.5,132.9,49.9,68.6,77.4,2.4,37.1,35.1,NaN,2014
2,3,Germany,192.7,112.3,72.9,75.3,87.1,5.6,36.1,30.2,NaN,2014
3,4,Sweden,180.9,106.1,61.7,75.2,103.7,9.4,34.1,17.5,NaN,2014
4,5,Finland,178.9,97.9,70.8,68.7,103.3,7.8,37.0,16.7,NaN,2014


In [33]:
df.to_csv("data/quality_of_life_indices_by_country.csv", index=False)